# Lab 01 — Modelagem de sistemas físicos por equações diferenciais

**Unidade I — Modelagem e análise de sistemas físicos** · conteúdo 1.1 do PPC

**Objetivos:**
1. Obter modelos de sistemas mecânicos e elétricos por leis físicas;
2. Converter EDOs em funções de transferência e em espaço de estados;
3. Modelar o **motor CC**, planta de referência do curso;
4. Explorar a analogia eletromecânica.

**Referências:** Åström & Murray (FBS), cap. 3 · Ogata, caps. 2–3 · Felício.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. Sistema mecânico: massa–mola–amortecedor

$$m\ddot{x} + b\dot{x} + kx = F(t) \quad\Longrightarrow\quad G(s) = \frac{X(s)}{F(s)} = \frac{1}{ms^2 + bs + k}$$

In [ ]:
# parâmetros físicos
m = 1.0    # massa [kg]
b = 2.0    # coeficiente de atrito viscoso [N.s/m]
k = 20.0   # rigidez da mola [N/m]

G_mec = ct.tf([1], [m, b, k])
print(G_mec)
print("Polos:", ct.poles(G_mec))
ct.damp(G_mec)

In [ ]:
resp = ct.step_response(G_mec)
plt.figure(figsize=(8, 4))
plt.plot(resp.time, resp.outputs, lw=2)
plt.xlabel('Tempo [s]')
plt.ylabel('Posição x [m]')
plt.title('Massa–mola–amortecedor: resposta a F = 1 N (degrau)')
plt.grid(True)
plt.show()

**Interprete fisicamente:** o valor final é $1/k = 0{,}05$ m (equilíbrio força da mola × força aplicada);
a oscilação decai com a taxa $\zeta\omega_n = b/(2m)$.

## 2. Sistema elétrico: circuito RC

$$RC\,\dot{v}_o + v_o = v_i \quad\Longrightarrow\quad G(s) = \frac{1}{RCs + 1}$$

In [ ]:
R = 10e3    # resistência [ohm]
C_f = 100e-6  # capacitância [F]
tau_rc = R * C_f
print(f"Constante de tempo: tau = {tau_rc:.2f} s")

G_rc = ct.tf([1], [tau_rc, 1])
resp = ct.step_response(G_rc)

plt.figure(figsize=(8, 4))
plt.plot(resp.time, resp.outputs, lw=2)
plt.axvline(tau_rc, color='r', ls=':', label=r'$t = \tau = RC$')
plt.axhline(0.632, color='r', ls=':')
plt.xlabel('Tempo [s]')
plt.ylabel('$v_o$ [V]')
plt.title('Circuito RC: carga do capacitor (degrau de 1 V)')
plt.legend()
plt.grid(True)
plt.show()

## 3. Analogia eletromecânica: circuito RLC × massa–mola–amortecedor

O RLC série com saída no capacitor obedece a
$LC\,\ddot{v}_o + RC\,\dot{v}_o + v_o = v_i$ — a **mesma estrutura** do sistema mecânico:

| Mecânico | Elétrico |
|---|---|
| massa $m$ | indutância $L$ |
| atrito $b$ | resistência $R$ |
| rigidez $k$ | elastância $1/C$ |
| força $F$ | tensão $v_i$ |

In [ ]:
# escolhemos R, L, C para reproduzir a MESMA dinâmica normalizada do sistema mecânico
L_e = 1.0
C_e = 1.0 / k          # 1/C  <->  k
R_e = b                # R    <->  b

G_rlc = ct.tf([1], [L_e * C_e, R_e * C_e, 1])
print("Polos do RLC:", ct.poles(G_rlc))
print("Polos do mecânico:", ct.poles(G_mec))

Os **polos são idênticos** — a dinâmica é a mesma (a menos do ganho estático).
Isto justifica estudar "sistemas de 2ª ordem" de forma unificada (Unidade II).

## 4. Motor CC — planta de referência do curso

$$L_a \dot{i}_a + R_a i_a + K_e\,\omega = v_a \qquad\qquad J\dot{\omega} + b_m\,\omega = K_t\, i_a$$

Estados: $x = [i_a,\ \omega]^T$ · entrada: tensão de armadura $v_a$ · saída: velocidade $\omega$.

$$A = \begin{bmatrix} -R_a/L_a & -K_e/L_a \\ K_t/J & -b_m/J \end{bmatrix},\quad
  B = \begin{bmatrix} 1/L_a \\ 0 \end{bmatrix},\quad
  C = \begin{bmatrix} 0 & 1 \end{bmatrix},\quad D = 0$$

In [ ]:
# parâmetros de um pequeno motor CC (valores didáticos, ordem de grandeza realista)
Ra = 2.0      # resistência de armadura [ohm]
La = 0.5      # indutância de armadura [H]
Kt = 0.1      # constante de torque [N.m/A]
Ke = 0.1      # constante de f.c.e.m. [V.s/rad]
J = 0.02      # inércia do rotor [kg.m^2]
bm = 0.005    # atrito viscoso [N.m.s/rad]

A = [[-Ra / La, -Ke / La],
     [Kt / J, -bm / J]]
B = [[1 / La],
     [0]]
C_out = [[0, 1]]   # medimos a velocidade omega
D = [[0]]

motor = ct.ss(A, B, C_out, D,
              inputs='va', outputs='omega',
              states=['ia', 'omega'], name='motor_cc')
print(motor)

# função de transferência equivalente Va -> Omega
G_motor = ct.tf(motor)
print("G_motor(s) =", G_motor)
print("Polos:", ct.poles(motor))

In [ ]:
resp = ct.step_response(motor)

plt.figure(figsize=(8, 4))
plt.plot(resp.time, resp.outputs[0], lw=2)
plt.xlabel('Tempo [s]')
plt.ylabel(r'$\omega$ [rad/s]')
plt.title('Motor CC: velocidade para degrau de 1 V na armadura')
plt.grid(True)
plt.show()

print("Ganho estático (rad/s por volt):", ct.dcgain(motor))

## 5. Redução de ordem: quando $L_a$ é desprezível

Em muitos motores pequenos a dinâmica elétrica é muito mais rápida que a mecânica.
Fazendo $L_a \to 0$: o modelo reduz-se a 1ª ordem

$$G(s) \approx \frac{K_m}{\tau_m s + 1},\qquad
  K_m = \frac{K_t}{R_a b_m + K_t K_e},\qquad
  \tau_m = \frac{R_a J}{R_a b_m + K_t K_e}$$

In [ ]:
Km = Kt / (Ra * bm + Kt * Ke)
tau_m = Ra * J / (Ra * bm + Kt * Ke)
print(f"Modelo reduzido: K = {Km:.2f}, tau = {tau_m:.3f} s")

G_red = ct.tf([Km], [tau_m, 1])

# comparação entre modelo completo (2ª ordem) e reduzido (1ª ordem)
t = np.linspace(0, 5, 500)
r2 = ct.step_response(motor, t)
r1 = ct.step_response(G_red, t)

plt.figure(figsize=(8, 4))
plt.plot(r2.time, r2.outputs[0], lw=2, label='modelo completo (2ª ordem)')
plt.plot(r1.time, r1.outputs, '--', lw=2, label='modelo reduzido (1ª ordem)')
plt.xlabel('Tempo [s]')
plt.ylabel(r'$\omega$ [rad/s]')
plt.title('Validade da redução de ordem do motor CC')
plt.legend()
plt.grid(True)
plt.show()

A concordância é excelente — usar o modelo de 1ª ordem simplificará a identificação
e a sintonia sem perda relevante de precisão. **Guardar essa conclusão para o projeto final.**

## 6. Modelagem não linear e linearização (metodologia CDS 110/Caltech)

Sistemas físicos reais são não lineares. O fluxo profissional (usado no curso CDS 110 do
Caltech, que acompanha o livro de Åström & Murray) é:
**modelo não linear → ponto de equilíbrio → linearização → análise linear**.

Exemplo: servomecanismo com braço acionado por motor contra uma mola
(a força da mola é não linear pela cinemática do mecanismo):
$$J\ddot{\theta} = -b\dot{\theta} - k r \sin\theta + \tau_m, \qquad y = l\,\theta$$

Em `python-control`, modelos não lineares são objetos `nlsys`, definidos pela função de
atualização de estado $\dot{x} = f(t, x, u)$. **Nomear sinais e usar o dicionário `params`**
torna o modelo autodocumentado e permite variar parâmetros sem reconstruí-lo:

In [ ]:
servo_params = {
    'J': 100.0,   # momento de inercia do motor
    'b': 10.0,    # amortecimento angular do braco
    'k': 1.0,     # constante da mola
    'r': 1.0,     # ponto de contato da mola no braco
    'l': 2.0,     # distancia ate a extremidade (saida)
}

def servo_update(t, x, u, params):
    """Dinamica nao linear do servomecanismo: x = [theta, thdot]."""
    J, b, k, r = map(params.get, ['J', 'b', 'k', 'r'])
    return np.array([
        x[1],
        (-b * x[1] - k * r * np.sin(x[0]) + u[0]) / J,
    ])

def servo_output(t, x, u, params):
    return np.array([params['l'] * x[0]])

servo = ct.nlsys(
    servo_update, servo_output, params=servo_params, name='servo',
    states=['theta', 'thdot'], inputs=['tau'], outputs=['y'])
print(servo)

**Passo 1 — ponto de equilíbrio.** Queremos operar em $\theta_e = 15°$. `find_eqpt`
resolve $f(x_e, u_e) = 0$ numericamente (aqui também dá para verificar à mão:
$u_e = k r \sin\theta_e$):

In [ ]:
theta_e = np.deg2rad(15)
xe, ue = ct.find_eqpt(servo, [theta_e, 0], 0.1, y0=servo_params['l'] * theta_e)
print(f"x_e = {xe},  u_e = {ue}")
print(f"verificacao analitica: u_e = k r sin(theta_e) = {np.sin(theta_e):.4f}")

**Passo 2 — linearização.** `linearize` calcula o jacobiano em $(x_e, u_e)$ e devolve um
sistema linear em espaço de estados, válido para pequenos desvios em torno do equilíbrio:

In [ ]:
P_lin = servo.linearize(xe, ue)
print(P_lin)
print("Polos:", P_lin.poles())

**Passo 3 — validar a linearização.** Comparamos o modelo linear com o não linear para um
desvio pequeno e um grande de torque em torno de $u_e$:

In [ ]:
t_nl = np.linspace(0, 60, 600)
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, du in zip(axs, [0.05, 0.8]):
    resp_nl = ct.input_output_response(servo, t_nl, ue + du, X0=xe)
    resp_li = ct.forced_response(P_lin, t_nl, du * np.ones_like(t_nl))
    ax.plot(resp_nl.time, resp_nl.outputs[0], lw=2, label='não linear')
    ax.plot(resp_li.time, servo_params['l'] * theta_e + resp_li.outputs, '--',
            lw=2, label='linearizado + offset')
    ax.set_title(f'desvio de torque Δu = {du}')
    ax.set_xlabel('Tempo [s]'); ax.grid(True); ax.legend()
axs[0].set_ylabel('y [m]')
fig.suptitle('A linearização vale localmente: boa para desvios pequenos')
plt.show()

Para desvio pequeno as curvas coincidem; para desvio grande a não-linearidade do seno aparece.
**Este fluxo (nlsys → find_eqpt → linearize) será usado no projeto final** para obter o modelo
de projeto a partir da física da planta escolhida.

---

> **Conexão com a bancada:** o motor CC modelado nesta aula é a versão idealizada do
> **kit físico do laboratório** (motoredutor com encoder de quadratura + ponte H L298N +
> Arduino) que será usado no projeto final. Lá, os parâmetros $K_m$ e $\tau_m$ não serão
> dados: serão **identificados** com os métodos dos Labs 02, 03, 07 e 09. Guarde este
> notebook — ele é o gabarito conceitual do que você vai medir.

> **🖼️ Figuras de apoio nos livros:**
> - Nise, **Figura 2.15** — sistema massa–mola–amortecedor e sua FT (exemplo 'Uma Equação de Movimento'). Cap. 2, §2.5, p. 107–108 do arquivo PDF (a cópia digital não exibe o nº impresso).
> - Nise, **Figura 2.35** — servomotor CC controlado pela armadura: esquema e diagrama para a dedução da FT. Cap. 2, seção de sistemas eletromecânicos, p. 127–128 do arquivo PDF (a cópia digital não exibe o nº impresso).
> - Ogata, **Figura 5.1** — diagramas de blocos do sistema de 1ª ordem. Cap. 5, §5.2, **p. 147** (p. 158 do PDF).

## Exercícios (relatório do Lab 01)

**E1.** Modele um **tanque de nível** com área $A_t = 0{,}5$ m², vazão de entrada $q_i$ (entrada)
e vazão de saída linearizada $q_o = h/R_v$ com $R_v = 100$ s/m². Obtenha $G(s) = H(s)/Q_i(s)$,
identifique $K$ e $\tau$ e simule o degrau.

**E2.** No massa–mola–amortecedor, varie $b \in \{0{,}5;\ 2;\ 8{,}9;\ 20\}$ mantendo $m$ e $k$.
Trace as quatro respostas ao degrau em um gráfico e classifique o amortecimento de cada caso
(use `ct.damp`). Qual valor de $b$ dá amortecimento crítico?

**E3.** Para o motor CC, mude a saída para a **corrente** $i_a$ ($C = [1\ \ 0]$) e simule o degrau.
Explique fisicamente o pico inicial de corrente e seu valor de regime.

**E4.** Obtenha a FT do motor a partir das matrizes usando álgebra:
$G(s) = C(sI - A)^{-1}B$ com `sympy` ou numericamente em uma frequência de teste, e compare
com `ct.tf(motor)`.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui